In [1]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "de_helpers.py").exists())
sys.path.insert(0, str(ROOT / "src"))

# Databricks Connect session via Databricks CLI authentication
from de_helpers import (
    q, q1, spark_q, assert_grain, tables, columns, explain, display, fs_ls,
    get_spark, get_dbutils, spark_read, plan_string, has_plan_operator
)
from pyspark.sql import functions as F, Window

spark = get_spark()
dbutils = get_dbutils()


In [9]:
# Lists the silver folder (/Volumes/workspace/iraonfridays/de_prep/silver)
fs_ls()

,name,size_kb,size,modificationTime,path
0,claim_services.parquet,466.70,477920,1791380835000,/Volumes/workspace/iraonfridays/de_prep/silver...
1,claims.parquet,416.50,426449,1791380835000,/Volumes/workspace/iraonfridays/de_prep/silver...
2,diagnoses.parquet,1.90,1934,1791380835000,/Volumes/workspace/iraonfridays/de_prep/silver...
3,members.parquet,27.80,28426,1791380835000,/Volumes/workspace/iraonfridays/de_prep/silver...
4,payments.parquet,190.40,194982,1791380834000,/Volumes/workspace/iraonfridays/de_prep/silver...
5,plans.parquet,0.70,756,1791380834000,/Volumes/workspace/iraonfridays/de_prep/silver...
6,procedures.parquet,2.40,2473,1791380833000,/Volumes/workspace/iraonfridays/de_prep/silver...
7,provider_specialty_history.parquet,1.80,1812,1791380833000,/Volumes/workspace/iraonfridays/de_prep/silver...
8,providers.parquet,7.20,7413,1791380835000,/Volumes/workspace/iraonfridays/de_prep/silver...
9,state_lookup.parquet,1.10,1084,1791380835000,/Volumes/workspace/iraonfridays/de_prep/silver...


In [12]:
# Check root of your volume
fs_ls("/Volumes/workspace/iraonfridays/de_prep/silver")

# # Or with dbutils.fs.ls directly:
# items = dbutils.fs.ls("/Volumes/workspace/iraonfridays/de_prep/silver")
# display(items)


,name,size_kb,size,modificationTime,path
0,claim_services.parquet,466.70,477920,1791380835000,/Volumes/workspace/iraonfridays/de_prep/silver...
1,claims.parquet,416.50,426449,1791380835000,/Volumes/workspace/iraonfridays/de_prep/silver...
2,diagnoses.parquet,1.90,1934,1791380835000,/Volumes/workspace/iraonfridays/de_prep/silver...
3,members.parquet,27.80,28426,1791380835000,/Volumes/workspace/iraonfridays/de_prep/silver...
4,payments.parquet,190.40,194982,1791380834000,/Volumes/workspace/iraonfridays/de_prep/silver...
5,plans.parquet,0.70,756,1791380834000,/Volumes/workspace/iraonfridays/de_prep/silver...
6,procedures.parquet,2.40,2473,1791380833000,/Volumes/workspace/iraonfridays/de_prep/silver...
7,provider_specialty_history.parquet,1.80,1812,1791380833000,/Volumes/workspace/iraonfridays/de_prep/silver...
8,providers.parquet,7.20,7413,1791380835000,/Volumes/workspace/iraonfridays/de_prep/silver...
9,state_lookup.parquet,1.10,1084,1791380835000,/Volumes/workspace/iraonfridays/de_prep/silver...


In [ ]:
# highest paid member_id


In [15]:
df = q('''
select *
from claims
''')
df

,claim_id,member_id,provider_id,claim_date,submitted_date,updated_at,status,claim_type,primary_diagnosis_code,billed_amount,paid_amount,ingested_at,source_system,_ingest_batch,_loaded_at
0,CLM0000024,MBR01086,PRV0025,2024-01-06,2024-01-10,2024-02-14 21:11:00,Approved,Professional,O80,"2,307.76","1,138.33",2024-02-16 21:11:00,CLAIMS_API,2025-07-01T02:05:00Z,2026-10-07 20:47:42
1,CLM0000047,MBR00615,PRV0144,2024-01-12,2024-01-14,2024-01-19 15:39:00,Approved,Professional,J45.909,505.93,262.61,2024-01-19 15:39:00,CLAIMS_API,2025-07-01T02:05:00Z,2026-10-07 20:47:42
2,CLM0000063,MBR00209,PRV0049,2024-01-18,2024-01-19,2024-01-25 01:34:00,Submitted,Professional,F41.1,"8,010.45",NaN,2024-01-26 01:34:00,CLAIMS_API,2025-07-01T02:05:00Z,2026-10-07 20:47:42
3,CLM0000067,MBR00328,PRV0113,2024-01-19,2024-01-25,2024-02-06 07:13:00,Approved,Professional,K21.9,"7,032.27","3,002.39",2024-02-09 07:13:00,CLAIMS_API,2025-07-01T02:05:00Z,2026-10-07 20:47:42
4,CLM0000081,MBR00620,PRV0054,2024-01-24,2024-01-24,2024-02-11 08:38:00,Approved,Professional,E66.9,"2,739.30","1,903.22",2024-02-12 08:38:00,CLAIMS_API,2025-07-01T02:05:00Z,2026-10-07 20:47:42
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11976,CLM0011951,MBR00923,PRV0024,2025-06-17,2025-06-24,2025-07-11 06:06:00,Submitted,Professional,M17.11,352.58,NaN,2025-07-14 06:06:00,CLAIMS_API,2025-07-01T02:05:00Z,2026-10-07 20:47:42
11977,CLM0011955,MBR00041,PRV0135,2025-06-03,2025-06-05,2025-07-19 14:38:00,Approved,Institutional,N39.0,129.65,70.45,2025-07-21 14:38:00,CLAIMS_API,2025-07-01T02:05:00Z,2026-10-07 20:47:42
11978,CLM0011956,MBR00950,PRV0100,2025-06-10,2025-06-11,2025-07-16 22:44:00,Approved,Professional,E78.5,"6,528.46","3,215.21",2025-07-19 22:44:00,CLAIMS_API,2025-07-01T02:05:00Z,2026-10-07 20:47:42
11979,CLM0011972,MBR00253,PRV0184,2025-06-06,2025-06-12,2025-07-21 11:39:00,Approved,Professional,Z79.899,"1,456.42","1,116.41",2025-07-23 11:39:00,CLAIMS_API,2025-07-01T02:05:00Z,2026-10-07 20:47:42


In [16]:
# Q: Show all the duplicate rows, including the original
# GROUP BY collapses rows, so use a window count to keep the full records.

all_duplicates = q('''
select * 
from (
    select *, count(*) over (partition by claim_id) as cnt
    from claim_services
) t
where cnt > 1
order by claim_id
''')

all_duplicates

,claim_id,service_id,procedure_code,service_date,units,service_amount,claim_type,cnt
0,CLM0000001,1,96413,2024-01-19,1,"3,103.68",Institutional,3
1,CLM0000001,2,77385,2024-01-18,1,"2,990.01",Institutional,3
2,CLM0000001,3,96413,2024-01-21,1,"2,829.22",Institutional,3
3,CLM0000002,1,96413,2024-01-21,2,"6,592.18",Professional,2
4,CLM0000002,2,96413,2024-01-23,1,"3,397.26",Professional,2
...,...,...,...,...,...,...,...,...
35759,CLM0011999,1,77385,2025-06-26,1,"2,407.40",Professional,3
35760,CLM0012000,1,90935,2025-06-22,1,702.31,Professional,2
35761,CLM0012000,2,90935,2025-06-22,1,447.24,Professional,2
35762,CLM0012001,1,83036,2025-06-26,1,51.00,Institutional,2


In [ ]:
#Code to practice
# Write each block from memory once before the interview; these versions match every answer above.
# 1. Validate, deduplicate and build the ledger in one pass.
#  Split invalid rows out before ranking, or an invalid row can win rn = 1 and
#  push the good row into the duplicates.

from pyspark.sql import functions as F, Window

checked = landing.withColumn("rejection_code",
    F.when(F.col("claim_id").isNull(), "ERR_NULL_PK")
     .when(F.col("amount") < 0, "ERR_NEGATIVE_AMOUNT")
     .when(F.col("service_date") > F.current_date(), "ERR_FUTURE_SERVICE_DATE"))

quarantined = checked.filter(F.col("rejection_code").isNotNull())
valid = checked.filter(F.col("rejection_code").isNull())

w = Window.partitionBy("claim_id").orderBy(
    F.col("updated_at").desc(),
    F.col("_source_file").desc(),
    F.col("_file_modified_at").desc())   # same values on every rerun
ranked = valid.withColumn("rn", F.row_number().over(w))

labelled = (ranked
    .withColumn("bucket", F.when(F.col("rn") == 1, "curated").otherwise("duplicate"))
    .unionByName(quarantined.withColumn("bucket", F.lit("quarantined")),
                 allowMissingColumns=True))

ledger = labelled.groupBy("bucket").agg(F.count("*").alias("rows"),
                                        F.sum("amount").alias("dollars"))
curated = ranked.filter("rn = 1").drop("rn")   # goes to the MERGE

In [ ]:
from pyspark.sql import functions as F, Window
from pyspark import sparksession

spark = sparksession.build('dv')

checked = (landing
            .withColumn("rejection_code",
            F.when(F.col("claim_id").isNull(), "ERR_NULL_PK")
            .when(F.col("amount")<0, "ERR_NEGATIVE_AMOUNT")
            .when(F.col("service_date")> F.current_date(), "ERR_FUTURE_SERVICE_DATE")
            )
    )

quarantined = checked.filter(F.col("rejection_code").isNotNull())
valid = checked.filter(F.col("rejection_code").isNull())

w = Window.partitionBy("claim_id").orderBy(
    F.col("updated_at").desc(),
    F.col("_source_file").desc(),
    F.col("_file_modified_at").desc()
)

ranked = valid.withColumn("rn", F.row_number().over(w))

labelled = (
    ranked.withColumn("bucket", F.when(F.col("rn")==1, "curated").otherwise("duplicate"))
    .unionByName(quarantined.withColumn("bucket", F.lit("quarantined")), allowMissingColumns=True)
)

ledger = labelled.groupBy("bucket").agg(F.count("*").alias("rows"), F.sum("amount").alias("dollars"))

curated = ranked.filter("rn=1").drop("rn")

In [ ]:
# Domain check: an 837 file carries billed charges, not paid amounts (payment comes from adjudication; the 835 is the remittance). If your landing feed was 837, your ledger balanced billed dollars. Know which column it was.
# 2. MERGE

df = q('''
    MERGE INTO silver.claims t
    USING curated s
    ON  t.claim_id = s.claim_id
    AND t.service_year_month = s.service_year_month
    WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *;
''')

df

In [ ]:
from asyncio import queues
# 3. Grain-safe join and grain test
df = q('''
    WITH lines AS (
    SELECT claim_id, COUNT(*) AS line_count, SUM(line_billed_amount) AS line_billed
    FROM silver.claim_services
    GROUP BY claim_id
    )
    SELECT c.provider_id,
        COUNT(*)           AS claims,
        SUM(c.paid_amount) AS paid,
        SUM(l.line_count)  AS lines
    FROM silver.claims c
    LEFT JOIN lines l ON l.claim_id = c.claim_id
    GROUP BY c.provider_id;

    -- must return no rows
    SELECT claim_id, line_number, COUNT(*)
    FROM silver.claim_services
    GROUP BY claim_id, line_number
    HAVING COUNT(*) > 1;
''')

In [ ]:
# 4. Broadcast join and AQE
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

joined = claims.join(F.broadcast(providers), "provider_id")
joined.explain()   # expect BroadcastHashJoin, no Exchange on the providers side